# MOD-04 · NB-02 — Decision Trees & Random Forest
### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
---
**Learning objectives**
- Build, visualise and interpret a clinical decision tree
- Understand bias-variance tradeoff via tree depth experiments
- Fit a Random Forest and tune hyperparameters (n_estimators, max_depth, min_samples_leaf)
- Extract and plot feature importances (MDI and permutation)
- Compare Decision Tree vs Random Forest on clinical holdout

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `sklearn`, `pandas`, `numpy`, `matplotlib`


## 1. Setup & Shared Dataset

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              RocCurveDisplay, ConfusionMatrixDisplay,
                              brier_score_loss)
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04",exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

# ── Dataset (same seed as NB-01) ──────────────────────────────────────────────
np.random.seed(42); N=3000
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,4)); np.random.seed(99)
age=np.random.normal(62,15,N).clip(18,92).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
payer=np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type=np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days=np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5+0.02*(age-62)/15),N)
hypertension=np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity=np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression=np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb=diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose=np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine=np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c=np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp=np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi=np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
       +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
       +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit),N)
df=pd.DataFrame({
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,
    "ckd":ckd,"obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,
    "readmitted":readmitted,
})
df["los_gt7"]=(df.los_days>7).astype(int)
NUMERIC=["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY=["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC=["sex","payer","admit_type"]
FEATURES=NUMERIC+BINARY+CATEGORIC; TARGET="readmitted"

# ── Preprocessing + split ─────────────────────────────────────────────────────
pre = ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),NUMERIC),
    ("bin",SimpleImputer(strategy="most_frequent"),BINARY),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),CATEGORIC),
])
X=df[FEATURES]; y=df[TARGET]
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
pre.fit(X_tr)
Xtr=pre.transform(X_tr); Xte=pre.transform(X_te)
ohe_names=pre.named_transformers_["cat"]["ohe"].get_feature_names_out(CATEGORIC)
feat_names=NUMERIC+BINARY+list(ohe_names)
print(f"Train: {Xtr.shape} | Test: {Xte.shape} | Positive rate: {y_tr.mean()*100:.1f}%")


## 2. Decision Tree — Building Intuition

In [ ]:
# Shallow tree (depth=4) for interpretability
dt_shallow = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
dt_shallow.fit(Xtr, y_tr)
prob_dt = dt_shallow.predict_proba(Xte)[:,1]
auc_dt  = roc_auc_score(y_te, prob_dt)
print(f"Decision Tree (depth=4): AUC = {auc_dt:.4f}")

# Visualise the tree
fig, ax = plt.subplots(figsize=(22,10))
plot_tree(dt_shallow, feature_names=feat_names, class_names=["Not readmitted","Readmitted"],
          filled=True, rounded=True, fontsize=8, ax=ax,
          proportion=False, impurity=True)
ax.set_title("Decision Tree (max_depth=4) — 30-day readmission prediction", fontsize=13)
plt.tight_layout(); plt.savefig("/tmp/mod04/decision_tree.png",bbox_inches="tight",dpi=150); plt.show()


In [ ]:
# Text representation — easier for clinical reporting
tree_rules = export_text(dt_shallow, feature_names=feat_names)
print("Decision tree rules (first 60 lines):")
print("\n".join(tree_rules.split("\n")[:60]))


## 3. Bias-Variance Tradeoff — Tree Depth Experiment

In [ ]:
depths = range(1, 21)
train_aucs, test_aucs = [], []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=42)
    cv = cross_val_score(dt, Xtr, y_tr, cv=skf, scoring="roc_auc")
    train_aucs.append(dt.fit(Xtr,y_tr).score(Xtr,y_tr))
    test_aucs.append(cv.mean())

best_depth = depths[np.argmax(test_aucs)]
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(depths, [dt_shallow.score(Xtr,y_tr)]*len(depths), "b--", alpha=0.3)
ax.plot(depths, test_aucs, "o-", color="#1F4E79", lw=2, label="CV AUC (5-fold)")
ax.axvline(best_depth, color="red", ls="--", lw=1.5, label=f"Optimal depth={best_depth}")
ax.axvline(4, color="orange", ls=":", lw=1.5, label="Depth=4 (interpretable)")
ax.fill_between(depths, [v-0.01 for v in test_aucs], [v+0.01 for v in test_aucs], alpha=0.1, color="#1F4E79")
ax.set_xlabel("Tree depth"); ax.set_ylabel("CV AUC-ROC")
ax.set_title("Bias-variance tradeoff — tree depth vs CV AUC")
ax.legend(fontsize=9); ax.set_ylim(0.5, 0.85)
plt.tight_layout(); plt.savefig("/tmp/mod04/tree_depth.png",bbox_inches="tight"); plt.show()
print(f"Best CV depth: {best_depth} (AUC={max(test_aucs):.4f})")
print(f"Deep trees overfit: train accuracy keeps rising while CV AUC plateaus/drops")


## 4. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=10,
                            max_features="sqrt", class_weight="balanced",
                            random_state=42, n_jobs=-1)
rf.fit(Xtr, y_tr)
prob_rf = rf.predict_proba(Xte)[:,1]
auc_rf  = roc_auc_score(y_te, prob_rf)
ap_rf   = average_precision_score(y_te, prob_rf)
brier_rf = brier_score_loss(y_te, prob_rf)
print(f"Random Forest: AUC={auc_rf:.4f} | Avg Precision={ap_rf:.4f} | Brier={brier_rf:.4f}")
print(f"Decision Tree (depth=4): AUC={auc_dt:.4f}")
print(f"AUC improvement: {auc_rf-auc_dt:+.4f}")


## 5. Feature Importance — MDI vs Permutation

In [ ]:
# MDI (Mean Decrease Impurity) — built-in, fast but biased toward high-cardinality features
mdi_imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)

# Permutation importance — model-agnostic, unbiased
np.random.seed(42)
perm_result = permutation_importance(rf, Xte, y_te, n_repeats=10,
                                      random_state=42, scoring="roc_auc")
perm_imp = pd.Series(perm_result.importances_mean, index=feat_names).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1,2,figsize=(14,6))
for ax, imp, title, color in [
    (axes[0], mdi_imp, "MDI Feature Importance
(built-in, biased toward high-cardinality)", "#4878CF"),
    (axes[1], perm_imp,"Permutation Importance
(model-agnostic, evaluated on test set)", "#D65F5F"),
]:
    ax.barh(imp.index[::-1], imp.values[::-1], color=color, alpha=0.85, edgecolor="white")
    ax.set_xlabel("Importance score"); ax.set_title(title, fontsize=10)
    for i,(name,val) in enumerate(zip(imp.index[::-1],imp.values[::-1])):
        ax.text(val+0.001, i, f"{val:.4f}", va="center", fontsize=8)

plt.tight_layout(); plt.savefig("/tmp/mod04/feature_importance.png",bbox_inches="tight"); plt.show()
print("Top 5 features (permutation importance):")
for name,val in perm_imp.head(5).items():
    print(f"  {name:20s}: {val:.4f}")


## 6. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    "n_estimators" : [100, 200],
    "max_depth"    : [6, 10, None],
    "min_samples_leaf": [5, 15],
    "max_features" : ["sqrt", "log2"],
}
rf_cv = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
gs = GridSearchCV(rf_cv, param_grid, cv=StratifiedKFold(3, shuffle=True, random_state=42),
                  scoring="roc_auc", n_jobs=-1, verbose=0)
gs.fit(Xtr, y_tr)

best_rf = gs.best_estimator_
prob_best = best_rf.predict_proba(Xte)[:,1]
auc_best  = roc_auc_score(y_te, prob_best)

print(f"Best parameters: {gs.best_params_}")
print(f"CV AUC (best): {gs.best_score_:.4f}")
print(f"Test AUC (best RF): {auc_best:.4f}")
print(f"Test AUC (default RF): {auc_rf:.4f}")


## 7. ROC Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42)
lr.fit(Xtr,y_tr); prob_lr=lr.predict_proba(Xte)[:,1]

fig,ax=plt.subplots(figsize=(8,6))
for prob, label, color in [
    (prob_lr, f"Logistic Regression (AUC={roc_auc_score(y_te,prob_lr):.3f})", "#6ACC65"),
    (prob_dt, f"Decision Tree d=4 (AUC={auc_dt:.3f})",                        "#B47CC7"),
    (prob_rf, f"Random Forest def (AUC={auc_rf:.3f})",                        "#4878CF"),
    (prob_best,f"RF tuned (AUC={auc_best:.3f})",                              "#D65F5F"),
]:
    from sklearn.metrics import roc_curve
    fpr,tpr,_ = roc_curve(y_te,prob)
    ax.plot(fpr,tpr,lw=2,label=label)
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlabel("1-Specificity"); ax.set_ylabel("Sensitivity")
ax.set_title("ROC curves — Decision Tree vs Random Forest variants")
ax.legend(fontsize=9,loc="lower right")
plt.tight_layout(); plt.savefig("/tmp/mod04/roc_comparison.png",bbox_inches="tight"); plt.show()


## 8. Knowledge Check
1. Why does a single deep decision tree overfit while a random forest with deep trees does not?
2. Permutation importance for `hf` is much lower than MDI importance. What might explain this discrepancy?
3. Your RF has 300 trees. A colleague says 1000 trees would always be better. Is this true?
4. The tree split on `creatinine > 1.45` at the root. Is this clinically plausible?
5. Re-train the RF with `class_weight="balanced_subsample"`. How does AUC change on the test set?


In [ ]:
# Exercise 5
rf_bs=RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced_subsample",random_state=42,n_jobs=-1)
rf_bs.fit(Xtr,y_tr)
auc_bs=roc_auc_score(y_te,rf_bs.predict_proba(Xte)[:,1])
print(f"RF balanced          AUC={auc_rf:.4f}")
print(f"RF balanced_subsample AUC={auc_bs:.4f}")
print(f"Difference: {auc_bs-auc_rf:+.4f}")


---
**Next → NB-03: XGBoost & LightGBM for Clinical Prediction**